# Sector ETF Technical Signal Comparison

Research question: Which technical trading signal performs most robustly across major U.S. sector ETFs, and does its performance persist out-of-sample relative to the S&P 500 benchmark?

We test Moving Average Crossover, RSI, and Momentum across five major U.S. sector ETFs. Parameters and sector choices are selected only on the 2010-2019 in-sample period using the Sortino ratio. Out-of-sample performance from 2020-2025 is used only for validation. All signals are lagged by one trading day to avoid look-ahead bias, and reported strategy results are net of 10 basis points one-way transaction costs. The S&P 500 is used as a benchmark rather than a traded signal candidate.

In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%matplotlib inline

import module
importlib.reload(module);

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})

In [ ]:
# Global settings
START_DATE = "2010-01-01"
END_DATE = "2025-12-31"
IS_END = "2019-12-31"
OOS_START = "2020-01-01"
TRADE_COST = 0.001  # 10 bps one-way transaction cost

sector_tickers = {
    "XLF": "Financials",
    "XLK": "Technology",
    "XLV": "Healthcare",
    "XLP": "Consumer Staples",
    "XLY": "Consumer Discretionary",
}
BENCHMARK = "^GSPC"
sector_etfs = list(sector_tickers.keys())

In [ ]:
# Data loading: use local CSV cache first, with yfinance only as a fallback.
from pathlib import Path

data_dir = Path("data")
sector_cache = data_dir / "sector_etfs.csv"
spx_cache = data_dir / "spx.csv"

if sector_cache.exists():
    df_sectors = pd.read_csv(sector_cache, index_col=0, parse_dates=True)
    df_sectors = df_sectors[[t for t in sector_etfs if t in df_sectors.columns]]
else:
    df_sectors = pd.DataFrame()

if spx_cache.exists():
    df_spx = pd.read_csv(spx_cache, index_col=0, parse_dates=True)
    df_spx = df_spx[[BENCHMARK]] if BENCHMARK in df_spx.columns else pd.DataFrame()
else:
    df_spx = pd.DataFrame()

missing = [t for t in sector_etfs if t not in df_sectors.columns]
if BENCHMARK not in df_spx.columns:
    missing.append(BENCHMARK)

if missing:
    try:
        import yfinance as yf
        raw = yf.download(missing, start=START_DATE, end=END_DATE, progress=False, auto_adjust=True)
        downloaded = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw
        if isinstance(downloaded, pd.Series):
            downloaded = downloaded.to_frame(missing[0])
        downloaded.index = pd.to_datetime(downloaded.index)
        for ticker in missing:
            if ticker in downloaded.columns:
                if ticker == BENCHMARK:
                    df_spx[ticker] = downloaded[ticker]
                else:
                    df_sectors[ticker] = downloaded[ticker]
    except Exception as exc:
        raise RuntimeError(f"Missing cached data for {missing} and yfinance fallback failed: {exc}")

df_indices = df_sectors[sector_etfs].join(df_spx[[BENCHMARK]], how="inner")
df_indices = df_indices.loc[START_DATE:END_DATE].ffill().dropna()
df_indices_is = df_indices.loc[:IS_END]
df_indices_oos = df_indices.loc[OOS_START:]

print(f"Data shape: {df_indices.shape}")
print(f"Date range: {df_indices.index[0].date()} to {df_indices.index[-1].date()}")
print(pd.Series({t: sector_tickers.get(t, "Benchmark") for t in df_indices.columns}, name="Label"))

## Methodology

Each signal is evaluated independently on each sector ETF. The signal is generated from information available at the close of day `t`, but the strategy earns the return on day `t+1`; this one-day lag prevents look-ahead bias. Parameter selection is based only on in-sample Sortino ratio. After a signal's best sector and parameters are chosen in-sample, the same frozen choice is evaluated out-of-sample.

In [ ]:
def ma_crossover_signal(price_series, short_window=50, long_window=200):
    return module.ma_signal(price_series, short_window, long_window)


def momentum_signal(price_series, lookback=126):
    price = price_series.copy()
    momentum = price / price.shift(lookback) - 1.0
    signal = (momentum > 0).astype(float).fillna(0.0)
    position_change = signal.diff().fillna(signal).abs()
    return pd.DataFrame({
        "price": price,
        "momentum": momentum,
        "signal": signal,
        "position_change": position_change,
    })


def donchian_signal(price_series, window=55):
    price = price_series.copy()
    upper = price.rolling(window=window).max().shift(1)
    lower = price.rolling(window=window).min().shift(1)
    signal = pd.Series(0.0, index=price.index)
    in_position = False

    for i in range(len(price)):
        if np.isnan(upper.iloc[i]) or np.isnan(lower.iloc[i]):
            signal.iloc[i] = 0.0
            continue
        if not in_position and price.iloc[i] > upper.iloc[i]:
            in_position = True
        elif in_position and price.iloc[i] < lower.iloc[i]:
            in_position = False
        signal.iloc[i] = 1.0 if in_position else 0.0

    position_change = signal.diff().fillna(signal).abs()
    return pd.DataFrame({
        "price": price,
        "upper": upper,
        "lower": lower,
        "signal": signal,
        "position_change": position_change,
    })


ma_grid = [
    {"short_window": short, "long_window": long}
    for short in [20, 50, 75]
    for long in [100, 150, 200, 250]
    if short < long
]

rsi_grid = [
    {"period": 14, "oversold": os_, "overbought": ob}
    for os_ in [20, 25, 30, 35, 40]
    for ob in [60, 65, 70, 75, 80]
    if os_ < ob
]

momentum_grid = [{"lookback": lb} for lb in [21, 63, 126, 189, 252]]
donchian_grid = [{"window": w} for w in [20, 40, 55, 75, 100, 150, 200]]

signal_tests = {
    "MA Crossover": {"fn": ma_crossover_signal, "grid": ma_grid},
    "RSI": {"fn": module.rsi_signal, "grid": rsi_grid},
    "Momentum": {"fn": momentum_signal, "grid": momentum_grid},
}

In [ ]:
def single_etf_portfolio_value(signal_fn, price_series, params, trade_cost=TRADE_COST):
    """
    Long-only strategy on one ETF/index.
    Uses yesterday's signal to earn today's return and deducts one-way trading costs.
    """
    price_series = price_series.dropna()
    px = price_series.to_numpy(dtype=float)
    daily_ret = np.concatenate(([0.0], px[1:] / px[:-1] - 1.0))

    sig = signal_fn(price_series, **params)
    raw_signal = sig["signal"].reindex(price_series.index).fillna(0.0).to_numpy(dtype=float)

    position = np.concatenate(([0.0], raw_signal[:-1]))
    previous_position = np.concatenate(([0.0], position[:-1]))
    trade = np.abs(position - previous_position)

    daily_gross = position * daily_ret
    daily_cost = trade * trade_cost
    daily_net = daily_gross - daily_cost

    gross_pv = np.cumprod(1.0 + daily_gross)
    net_pv = np.cumprod(1.0 + daily_net)
    return gross_pv, net_pv, daily_net, position


def performance_metrics_from_pv(pv, daily_returns=None):
    pv = np.asarray(pv, dtype=float)
    if daily_returns is None:
        daily_returns = np.concatenate(([0.0], pv[1:] / pv[:-1] - 1.0))
    daily_returns = np.asarray(daily_returns, dtype=float)
    eval_returns = daily_returns[1:] if len(daily_returns) > 1 else daily_returns

    return {
        "Net Return": pv[-1] / pv[0] - 1.0,
        "CAGR": module.compute_cagr(pv),
        "Ann. Volatility": module.compute_annual_volatility(eval_returns),
        "Sharpe": module.compute_sharpe(eval_returns),
        "Sortino": module.compute_sortino(eval_returns),
        "Calmar": module.compute_calmar(pv),
        "Max Drawdown": module.compute_max_drawdown(pv),
    }


def benchmark_metrics(price_series):
    px = price_series.dropna().to_numpy(dtype=float)
    pv = px / px[0]
    dr = np.concatenate(([0.0], px[1:] / px[:-1] - 1.0))
    return pv, performance_metrics_from_pv(pv, dr)


def optimise_signal_on_index(signal_fn, price_is, param_grid):
    best_score = -np.inf
    best_params = None
    best_pv = None

    for params in param_grid:
        try:
            _, net_pv, daily_net, _ = single_etf_portfolio_value(signal_fn, price_is, params)
            metrics = performance_metrics_from_pv(net_pv, daily_net)
            score = metrics["Sortino"]
            if not np.isnan(score) and score > best_score:
                best_score = score
                best_params = params
                best_pv = net_pv
        except Exception:
            continue

    return best_params, best_score, best_pv

In [ ]:
# Optimise every signal-sector pair in-sample, then validate the frozen choice out-of-sample.
results = []
curve_store = {}

for signal_name, signal_info in signal_tests.items():
    signal_fn = signal_info["fn"]
    param_grid = signal_info["grid"]

    for ticker in sector_etfs:
        price_is = df_indices_is[ticker].dropna()
        price_oos = df_indices_oos[ticker].dropna()

        best_params, is_sortino, net_is = optimise_signal_on_index(signal_fn, price_is, param_grid)
        if best_params is None:
            continue

        _, net_oos, daily_oos, position_oos = single_etf_portfolio_value(signal_fn, price_oos, best_params)
        m_is = performance_metrics_from_pv(net_is)
        m_oos = performance_metrics_from_pv(net_oos, daily_oos)

        key = (signal_name, ticker)
        curve_store[key] = {
            "is": pd.Series(net_is, index=price_is.index),
            "oos": pd.Series(net_oos, index=price_oos.index),
            "position_oos": pd.Series(position_oos, index=price_oos.index),
            "params": best_params,
        }

        results.append({
            "Signal": signal_name,
            "Ticker": ticker,
            "Sector": sector_tickers[ticker],
            "Best Params": best_params,
            "IS Sortino": m_is["Sortino"],
            "OOS Sortino": m_oos["Sortino"],
            "IS Sharpe": m_is["Sharpe"],
            "OOS Sharpe": m_oos["Sharpe"],
            "IS CAGR": m_is["CAGR"],
            "OOS CAGR": m_oos["CAGR"],
            "IS Max DD": m_is["Max Drawdown"],
            "OOS Max DD": m_oos["Max Drawdown"],
            "IS Net Return": m_is["Net Return"],
            "OOS Net Return": m_oos["Net Return"],
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["Signal", "IS Sortino"], ascending=[True, False]).reset_index(drop=True)

display_cols = [
    "Signal", "Ticker", "Sector", "Best Params", "IS Sortino", "OOS Sortino",
    "IS CAGR", "OOS CAGR", "IS Max DD", "OOS Max DD", "IS Net Return", "OOS Net Return"
]
results_df[display_cols]

In [ ]:
# Best sector per signal is selected using IS Sortino only.
best_by_signal = (
    results_df.sort_values("IS Sortino", ascending=False)
    .groupby("Signal", as_index=False)
    .head(1)
    .sort_values("IS Sortino", ascending=False)
    .reset_index(drop=True)
)

spx_is_pv, spx_is_metrics = benchmark_metrics(df_indices_is[BENCHMARK])
spx_oos_pv, spx_oos_metrics = benchmark_metrics(df_indices_oos[BENCHMARK])

benchmark_comparison = best_by_signal[[
    "Signal", "Ticker", "Sector", "Best Params", "OOS Sortino",
    "OOS Sharpe", "OOS CAGR", "OOS Max DD", "OOS Net Return"
]].copy()
benchmark_comparison["Benchmark OOS Sortino"] = spx_oos_metrics["Sortino"]
benchmark_comparison["Benchmark OOS Sharpe"] = spx_oos_metrics["Sharpe"]
benchmark_comparison["Benchmark OOS CAGR"] = spx_oos_metrics["CAGR"]
benchmark_comparison["Benchmark OOS Max DD"] = spx_oos_metrics["Max Drawdown"]
benchmark_comparison["Benchmark OOS Net Return"] = spx_oos_metrics["Net Return"]

print("Best sector per signal selected only from in-sample Sortino:")
display(best_by_signal[display_cols])

print("Out-of-sample comparison against S&P 500 buy-and-hold:")
display(benchmark_comparison)

In [ ]:
# Final IS-selected strategy: choose the single best signal-sector pair by IS Sortino.
final_row = best_by_signal.iloc[0]
final_key = (final_row["Signal"], final_row["Ticker"])
final_curve = curve_store[final_key]["oos"]
final_params = curve_store[final_key]["params"]

print("FINAL IS-SELECTED STRATEGY")
print("=" * 72)
print(f"Signal: {final_row['Signal']}")
print(f"Sector: {final_row['Sector']} ({final_row['Ticker']})")
print(f"Parameters: {final_params}")
print(f"OOS Sortino: {final_row['OOS Sortino']:.3f}")
print(f"OOS CAGR: {final_row['OOS CAGR']:.2%}")
print(f"OOS Max Drawdown: {final_row['OOS Max DD']:.2%}")
print(f"S&P 500 OOS Sortino: {spx_oos_metrics['Sortino']:.3f}")
print(f"S&P 500 OOS CAGR: {spx_oos_metrics['CAGR']:.2%}")
print(f"S&P 500 OOS Max Drawdown: {spx_oos_metrics['Max Drawdown']:.2%}")

In [ ]:
# OOS comparison plots
fig, axes = plt.subplots(3, 1, figsize=(13, 15))

sortino_pivot = results_df.pivot(index="Ticker", columns="Signal", values="OOS Sortino").loc[sector_etfs]
sortino_pivot.plot(kind="bar", ax=axes[0])
axes[0].axhline(spx_oos_metrics["Sortino"], color="black", linestyle="--", linewidth=1.2, label="S&P 500")
axes[0].set_title("Out-of-Sample Sortino by Signal and Sector")
axes[0].set_ylabel("Sortino")
axes[0].set_xlabel("")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].plot(df_indices_oos.index, spx_oos_pv, color="black", linestyle="--", label="S&P 500")
for _, row in best_by_signal.iterrows():
    key = (row["Signal"], row["Ticker"])
    curve = curve_store[key]["oos"]
    axes[1].plot(curve.index, curve.values, label=f"{row['Signal']} on {row['Ticker']}")
axes[1].set_title("OOS Equity Curves for IS-Selected Signal/Sector Pairs")
axes[1].set_ylabel("Portfolio Value")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].fill_between(
    df_indices_oos.index,
    module.compute_drawdown_series(spx_oos_pv) * 100,
    0,
    color="black",
    alpha=0.15,
    label="S&P 500",
)
for _, row in best_by_signal.iterrows():
    key = (row["Signal"], row["Ticker"])
    curve = curve_store[key]["oos"]
    axes[2].plot(curve.index, module.compute_drawdown_series(curve.values) * 100, label=f"{row['Signal']} on {row['Ticker']}")
axes[2].set_title("OOS Drawdowns")
axes[2].set_ylabel("Drawdown (%)")
axes[2].legend()
axes[2].grid(alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

In [ ]:
# Parameter neighbourhood robustness around each IS-selected result.
robustness_rows = []

for _, row in best_by_signal.iterrows():
    signal_name = row["Signal"]
    ticker = row["Ticker"]
    signal_info = signal_tests[signal_name]
    selected_score = row["IS Sortino"]
    scores = []

    for params in signal_info["grid"]:
        _, net_pv, _, _ = single_etf_portfolio_value(signal_info["fn"], df_indices_is[ticker], params)
        metrics = performance_metrics_from_pv(net_pv)
        if not np.isnan(metrics["Sortino"]):
            scores.append(metrics["Sortino"])

    scores = np.asarray(scores, dtype=float)
    robustness_rows.append({
        "Signal": signal_name,
        "Ticker": ticker,
        "Selected IS Sortino": selected_score,
        "Grid Median IS Sortino": np.nanmedian(scores),
        "Grid 75th Percentile": np.nanpercentile(scores, 75),
        "Grid Max": np.nanmax(scores),
        "Selected Minus Median": selected_score - np.nanmedian(scores),
    })

robustness_df = pd.DataFrame(robustness_rows)
robustness_df

In [ ]:
# Optional requested check: Donchian Channel on XLI, correctly labelled as Industrials.
XLI_TICKER = "XLI"

if XLI_TICKER in df_indices.columns:
    xli_prices = df_indices[XLI_TICKER].dropna()
elif (data_dir / "xli.csv").exists():
    xli_prices = pd.read_csv(data_dir / "xli.csv", index_col=0, parse_dates=True)[XLI_TICKER].dropna()
else:
    import yfinance as yf
    xli_prices = yf.download(XLI_TICKER, start=START_DATE, end=END_DATE, progress=False, auto_adjust=True)["Close"].dropna()

xli_is = xli_prices.loc[:IS_END]
xli_oos = xli_prices.loc[OOS_START:]

best_xli_donchian_params, best_xli_donchian_score, best_xli_donchian_is = optimise_signal_on_index(
    donchian_signal, xli_is, donchian_grid
)
_, xli_donchian_oos_pv, xli_donchian_oos_daily, _ = single_etf_portfolio_value(
    donchian_signal, xli_oos, best_xli_donchian_params
)

print("DONCHIAN CHANNEL ON XLI (Industrials)")
print("=" * 60)
print(f"Best Parameters: {best_xli_donchian_params}")
print(f"IS Sortino: {best_xli_donchian_score:.3f}")
print("OOS metrics:")
pd.Series(performance_metrics_from_pv(xli_donchian_oos_pv, xli_donchian_oos_daily))

## Discussion, Limitations, and Conclusion

The final selection rule is: choose using IS, judge using OOS. This prevents the out-of-sample period from influencing either the selected parameters or the chosen sector. Because all strategy returns use `position[t] = signal[t-1]`, the reported performance avoids same-day signal-return look-ahead bias.

Limitations include the small number of tested signal families, sensitivity to the chosen transaction cost assumption, and the fact that sector ETFs represent broad liquid exposures rather than individual security selection. The benchmark is a passive S&P 500 buy-and-hold series, so comparisons should distinguish between absolute return, risk-adjusted return, and drawdown control.

Use the printed final strategy cell above to complete the conclusion with the realised OOS Sortino, CAGR, maximum drawdown, and comparison against the S&P 500 benchmark.